In [1]:
import pandas as pd 
import numpy as np
import os 
import sqlite3

#### Data Extraction

In [2]:
# define relative paths
raw_dir = "../data/raw"
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

In [3]:
# extract raw data
df_docs = pd.read_parquet(os.path.join(raw_dir, "sales_documents.parquet"))
df_support = pd.read_parquet(os.path.join(raw_dir, "supporting_documents.parquet"))
df_comms = pd.read_parquet(os.path.join(raw_dir, "all_communications.parquet"))

bs_docs = pd.read_parquet(os.path.join(raw_dir, "business_documents.parquet"))
df_erp = pd.read_parquet(os.path.join(raw_dir, "erp_transactions.parquet"))
sales_items = pd.read_parquet(os.path.join(raw_dir, "sales_items.parquet"))

#### Create Events

In [4]:
# extract order creation events 
df_docs['timestamp'] = pd.to_datetime(df_docs['CREATIONDATE'])
events_creation = pd.DataFrame({
    'case_id': df_docs['SALESDOCUMENT'].astype(str),
    'activity':'ORder Created',
    'timestamp':df_docs['timestamp'],
    'user_or_role':'SAP_SYSTEM'
})

In [5]:
# Extract Fulfillment Events
df_support['timestamp'] = pd.to_datetime(df_support['ship_date']) 
events_fulfillment = pd.DataFrame({
    'case_id': df_support['order_number'].astype(str),
    'activity': df_support['document_type'].apply(lambda x: f"{str(x).upper()} Generated"),
    'timestamp': df_support['timestamp'],
    'user_or_role': df_support['carrier'].fillna('SYSTEM')
})

In [6]:
# Stream C: Communications Logs (The NLP Regex Fix)
df_comms['timestamp'] = pd.to_datetime(df_comms['timestamp'], format='mixed')

# FIX 1: Clean up missing roles so we don't get "Email: None"
df_comms['from_role'] = df_comms['from_role'].fillna('Unknown Role')

# FIX 2: Smarter Regex. SAP orders in this dataset start with 0. 
# We look for a '0' followed by exactly 9 digits.
order_pattern = r'\b(0\d{9})\b'
df_comms['extracted_order'] = df_comms['subject'].str.extract(order_pattern)
df_comms['extracted_order'] = df_comms['extracted_order'].fillna(df_comms['body'].str.extract(order_pattern)[0])

comms_with_orders = df_comms[df_comms['extracted_order'].notna()].copy()

events_comms = pd.DataFrame({
    'case_id': comms_with_orders['extracted_order'].astype(str), 
    'activity': comms_with_orders['from_role'].apply(lambda x: f"Email: {x}"),
    'timestamp': comms_with_orders['timestamp'],
    'user_or_role': comms_with_orders['from_name']
})

/tmp/ipykernel_46335/247538688.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_comms['extracted_order'] = df_comms['extracted_order'].fillna(df_comms['body'].str.extract(order_pattern)[0])


In [7]:
# df_erp.columns

#### Export Data

In [8]:
# # combine all streams into one master sequence 
# event_log = pd.concat([events_creation, events_fulfillment, events_comms], ignore_index=True)
# event_log = event_log.dropna(subset=['case_id', 'timestamp'])

In [9]:
# # sort chronologicaly 
# event_log = event_log.sort_values(by=['case_id', 'timestamp']).reset_index(drop=True)

In [10]:
# # load processed data 
# event_path = os.path.join(processed_dir, "event_log.parquet")
# event_log.to_parquet(event_path, index=False)

In [13]:
# 2. CONCATENATE & CLEAN
master_event_log = pd.concat([events_creation, events_fulfillment, events_comms], ignore_index=True)
master_event_log = master_event_log.dropna(subset=['case_id', 'timestamp'])

# FIX 1: Drop exact duplicate rows so they don't artificially inflate counts
master_event_log = master_event_log.drop_duplicates()

# FIX 2: The "Must Have Progress" Rule
# We group by case_id and count how many UNIQUE activities occurred. 
# If a case ONLY has "Order Created" (unique activities = 1), we drop it.
unique_activities = master_event_log.groupby('case_id')['activity'].nunique()
valid_cases = unique_activities[unique_activities > 1].index

# Keep only the valid cases that actually progressed through the business
master_event_log = master_event_log[master_event_log['case_id'].isin(valid_cases)]

# Sort chronologically to build the true process path
master_event_log = master_event_log.sort_values(by=['case_id', 'timestamp']).reset_index(drop=True)

print(f"Purged Ghost Orders. Clean event log created with {len(master_event_log)} total events.")
master_event_log.to_parquet(os.path.join(processed_dir, "advanced_event_log.parquet"), index=False)

Purged Ghost Orders. Clean event log created with 1616 total events.
